# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     | David Farias Rivera |
| **Fecha**      | 16/10/26  |
| **Expediente** | 754381  |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

- **Para poder entrenar y testear el modelo con datos diferentes, para asi detectar overfiting y modelos malos mas facil**

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

- **Entrenar con todos los datos da un mejor modelo, pero nos quedamos sin datos nuevos para saber si generaliza. La validación cruzada resuelve esto: usa todos los datos para entrenar y para probar, pero nunca los mismos al mismo tiempo.**


El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = pd.read_excel("/Users/david/Documents/Codigod/ITESO/01-ITESO/Lab 3/Motor Trend Car Road Tests.xlsx", header=0)
data = data.drop(columns=["model"])
print("Faltantes:", data.isna().sum().sum())
print("Duplicados:", data.duplicated().sum())

x = data.drop(columns=["mpg"])
y = data["mpg"]
data.shape


Faltantes: 0
Duplicados: 0


(32, 11)

In [11]:
loo = LeaveOneOut()
lr = LinearRegression()
mse = -cross_val_score(lr, x, y, cv=loo, scoring="neg_mean_squared_error")  # sklearn da el MSE negativo porque siempre maximiza

print("Modelos entrenados:", len(mse))
print("MSE promedio:", mse.mean())
print("Desviacion del MSE:", mse.std())


Modelos entrenados: 32
MSE promedio: 12.181558006901959
Desviacion del MSE: 17.06739987188856


In [12]:
lr.fit(x, y)
print("MSE train:", np.mean((y - lr.predict(x)) ** 2))
print("MSE prediciendo el promedio:", np.mean((y - np.mean(y)) ** 2)) 


MSE train: 4.60920093802033
MSE prediciendo el promedio: 35.188974609374995


In [13]:
# Pipeline para que el escalador solo aprenda de los 31 autos de entrenamiento en cada vuelta
for i in [0.1, 1, 10, 100]:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=i))
    mse_ridge = -cross_val_score(ridge, x, y, cv=loo, scoring="neg_mean_squared_error")
    print("Ridge alpha =", i, "MSE:", mse_ridge.mean())

for i in [0.01, 0.1, 0.5, 1]:
    lasso = make_pipeline(StandardScaler(), Lasso(alpha=i))
    mse_lasso = -cross_val_score(lasso, x, y, cv=loo, scoring="neg_mean_squared_error")
    print("Lasso alpha =", i, "MSE:", mse_lasso.mean())


Ridge alpha = 0.1 MSE: 11.605147726997913
Ridge alpha = 1 MSE: 9.452096328712823
Ridge alpha = 10 MSE: 7.352747596808642
Ridge alpha = 100 MSE: 10.914771099483513
Lasso alpha = 0.01 MSE: 11.442519572708541
Lasso alpha = 0.1 MSE: 9.39936361622801
Lasso alpha = 0.5 MSE: 8.314200983063266
Lasso alpha = 1 MSE: 8.358250818006702


Interpreta.

- **El modelo se equivoca en promedio 12.18 de MSE en autos que no vio, mucho peor que en train (4.6): hay sobreajuste por tener 10 variables y solo 32 autos. Ridge con alpha 10 lo baja a 7.35.**


## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [14]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Dataset Shape: (20640, 8) (20640,)
Dataset Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Dataset Target: ['MedHouseVal']


In [15]:
kf = KFold(n_splits=10, shuffle=True, random_state=0)  # Revolvemos porque los datos vienen ordenados por zona
lr = LinearRegression()
mse = -cross_val_score(lr, X, y, cv=kf, scoring="neg_mean_squared_error")

print("MSE de cada fold:", mse)
print("MSE promedio:", mse.mean())
print("Desviacion del MSE:", mse.std())

lr.fit(X, y)
mse_prom = np.mean((y - np.mean(y)) ** 2)
print("MSE train:", np.mean((y - lr.predict(X)) ** 2))
print("MSE prediciendo el promedio:", mse_prom)
print("R2 con validacion:", 1 - mse.mean() / mse_prom)  # Misma idea que 1 - RSS/TSS del lab 1

for i in [0.1, 1, 10, 100]:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=i))
    mse_ridge = -cross_val_score(ridge, X, y, cv=kf, scoring="neg_mean_squared_error")
    print("Ridge alpha =", i, "MSE:", mse_ridge.mean())


MSE de cada fold: [0.53625198 0.52258821 0.56926837 0.51329558 0.52696388 0.55357637
 0.53751507 0.5296315  0.51215682 0.47805147]
MSE promedio: 0.5279299259754939
Desviacion del MSE: 0.023475342583957667
MSE train: 0.5243209861846072
MSE prediciendo el promedio: 1.3315503000818074
R2 con validacion: 0.6035223559011934
Ridge alpha = 0.1 MSE: 0.527929699875162
Ridge alpha = 1 MSE: 0.5279277306361014
Ridge alpha = 10 MSE: 0.5279144425939004
Ridge alpha = 100 MSE: 0.5283365105812543


Interpreta.

- **El MSE es 0.528 en validación y 0.524 en train, casi igual, así que no hay sobreajuste y Ridge no ayuda. El R2 de 0.60 indica que le faltan variables al modelo.**


## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3